# DST Assignment 04 – Machine Learning Pipeline

In this assignment, you will build a complete Machine Learning classification pipeline. The dataset contains 10,000 student records.

**Core Task:** Implement the K-Nearest Neighbors (KNN) algorithm entirely from scratch using only NumPy. Then, compare your implementation against `scikit-learn` models.

> **Rules:**
> - You must use **exactly** the variable names requested in the `# TODO:` comments.
> - You may NOT use `sklearn.neighbors` or any other library for Task 4.
> - For random states, always use `random_state=42` to ensure your results match the autograder.

## Task 1: Data Loading & Exploration

In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# TODO: Load 'students_performance.csv' into `df`
df = pd.read_csv('students_performance.csv')

# TODO: Get the shape of the dataframe into `df_shape`
df_shape = df.shape

# TODO: Create a Series of missing value counts per column in `missing_counts`
missing_counts = df.isnull().sum()

# TODO: Create a Series containing the percentage distribution of the target 'grade' in `class_distribution`
class_distribution = df['grade'].value_counts(normalize=True) *100

print("df_shape:", df_shape)
print("missing_counts:",missing_counts)
print("class_distribution:", class_distribution)
df.head()

df_shape: (10000, 8)
missing_counts: student_id             0
study_hours            0
attendance_pct         0
previous_gpa         320
sleep_hours          300
extracurricular        0
tutoring_sessions      0
grade                  0
dtype: int64
class_distribution: grade
B    35.56
C    34.93
F    15.02
A    14.49
Name: proportion, dtype: float64


,student_id,study_hours,attendance_pct,previous_gpa,sleep_hours,extracurricular,tutoring_sessions,grade
0,1,8.61,91.9,3.69,7.14,No,6,A
1,2,5.08,53.0,2.50,4.51,No,4,C
2,3,15.00,92.8,3.34,7.55,No,7,A
3,4,2.40,54.6,1.46,7.11,No,2,F
4,5,4.13,77.9,1.97,7.68,No,2,C


## Task 2: Preprocessing

In [31]:
# TODO: Create a copy of df called `df_clean`
df_clean = df.copy()

# TODO: Fill missing values in 'previous_gpa' and 'sleep_hours' with the MEDIAN of their respective columns
df_clean['previous_gpa']=df_clean['previous_gpa'].fillna(df_clean['previous_gpa'].median())
df_clean['sleep_hours']=df_clean['sleep_hours'].fillna(df_clean['sleep_hours'].median())

# TODO: Create `label_map` dict to map 'extracurricular' ('Yes' -> 1, 'No' -> 0)
label_map = {'Yes': 1, 'No': 0}
# Apply the map to df_clean['extracurricular']
df_clean['extracurricular']=df_clean['extracurricular'].map(label_map)
# TODO: Drop the 'student_id' column (it should not be a feature)
df_clean=df_clean.drop(columns=['student_id'])
# TODO: Separate features into `X` (DataFrame) and target into `y` (Series)
X = df_clean.drop(columns=['grade'])
y = df_clean['grade']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nX.head():\n", X.head())
print("\ny.head():\n", y.head())

X shape: (10000, 6)
y shape: (10000,)

X.head():
    study_hours  attendance_pct  previous_gpa  sleep_hours  extracurricular  \
0         8.61            91.9          3.69         7.14                0   
1         5.08            53.0          2.50         4.51                0   
2        15.00            92.8          3.34         7.55                0   
3         2.40            54.6          1.46         7.11                0   
4         4.13            77.9          1.97         7.68                0   

   tutoring_sessions  
0                  6  
1                  4  
2                  7  
3                  2  
4                  2  

y.head():
 0    A
1    C
2    A
3    F
4    C
Name: grade, dtype: object


## Task 3: Feature Scaling & Train/Test Split

In [32]:
from sklearn.model_selection import train_test_split

# TODO: Implement manual standard scaling: Z = (X - mean) / std
# Calculate the mean and std for each column in X using NumPy or Pandas.
# Store the means in `scaler_mean` (Series or array) and stds in `scaler_std` (Series or array)
scaler_mean = X.mean()
scaler_std = X.std()

# TODO: Create `X_scaled` (DataFrame or numpy array) using your calculated mean/std
X_scaled = (X-scaler_mean)/scaler_std

# TODO: Split X_scaled and y into train and test sets (80/20 split, random_state=42)
# Use variables: X_train, X_test, y_train, y_test
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print("scalar_mean:\n",scaler_mean)
print("\nscaler_std:\n",scaler_std)
print("\nX_train shape:",X_train.shape)
print("\nX_test shape:", X_test.shape)

scalar_mean:
 study_hours           6.973721
attendance_pct       75.268230
previous_gpa          2.764480
sleep_hours           6.691763
extracurricular       0.512600
tutoring_sessions     3.548300
dtype: float64

scaler_std:
 study_hours           3.141908
attendance_pct       14.215914
previous_gpa          0.683436
sleep_hours           1.381401
extracurricular       0.499866
tutoring_sessions     1.776396
dtype: float64

X_train shape: (8000, 6)

X_test shape: (2000, 6)


## Task 4: KNN From Scratch (3 pts)
Do NOT use sklearn for this cell. Use pure NumPy.

In [33]:
from sklearn.metrics._plot.regression import PredictionErrorDisplay
def knn_predict(X_train_data, y_train_data, X_test_data, k=5):


    """
    Implement KNN classification.

    For each test sample:
    1. Compute Euclidean distances to all training samples
    2. Find the indices of the k nearest neighbors
       (Tie-break distances by picking the lower index first)
    3. Get the classes of those k neighbors
    4. Perform a majority vote to pick the predicted class
       (Tie-break votes by picking the alphabetically smallest label: 'A' < 'B' < 'C' < 'F')

    Args:
        X_train_data: numpy array of shape (n_train, n_features)
        y_train_data: numpy array of shape (n_train,) of string labels
        X_test_data: numpy array of shape (n_test, n_features)
        k: integer

    Returns:
        numpy array of predicted string labels (shape: n_test,)
    """
    # TODO: Implement this function from scratch
    X_train_data = np.asarray(X_train_data,dtype=float)
    X_test_data=np.asarray(X_test_data,dtype=float)
    y_train_data=np.asarray(y_train_data)

    predictions=[]
    for test_point in X_test_data:
        distances=np.sqrt(np.sum((X_train_data-test_point)**2, axis=1))
        nearest_indices=np.argsort(distances, kind='stable')[:k]
        nearest_labels=y_train_data[nearest_indices]

        labels,counts=np.unique(nearest_labels, return_counts=True)
        max_count=counts.max()
        tied_labels=sorted(labels[counts==max_count])
        predictions.append(tied_labels[0])

    return np.array(predictions)

# Convert to numpy arrays if they aren't already
X_train_np = np.array(X_train)
y_train_np = np.array(y_train)
X_test_np = np.array(X_test)
y_test_np = np.array(y_test)

# TODO: Use your function to predict with k=5, store in `knn_preds_k5`
knn_preds_k5 = knn_predict(X_train_np,y_train_np,X_test_np,k=5)

# TODO: Calculate accuracy for k=5, store in `knn_accuracy_k5`
knn_accuracy_k5 = (knn_preds_k5==y_test_np).mean()

# TODO: Use your function to predict with k=3, store in `knn_preds_k3`
knn_preds_k3 = knn_predict(X_train_np,y_train_np,X_test_np,k=3)

# TODO: Calculate accuracy for k=3, store in `knn_accuracy_k3`
knn_accuracy_k3 = (knn_preds_k3==y_test_np).mean()

print("knn_accuracy_k5:", knn_accuracy_k5)
print("knn_accuracy_k3:", knn_accuracy_k3)

knn_accuracy_k5: 0.834
knn_accuracy_k3: 0.821


## Task 5: Scikit-Learn Models
Compare your implementation against established algorithms.

In [35]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# TODO: Train Logistic Regression (max_iter=1000, random_state=42)
# Store model in `logreg_model`, accuracy in `logreg_accuracy`
logreg_model = LogisticRegression(max_iter=1000, random_state=42)
logreg_model.fit(X_train, y_train)
logreg_accuracy = accuracy_score(y_test, logreg_model.predict(X_test))

# TODO: Train Decision Tree (random_state=42)
# Store model in `dt_model`, accuracy in `dt_accuracy`
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
dt_accuracy = accuracy_score(y_test,dt_model.predict(X_test))

# TODO: Train Random Forest (n_estimators=100, random_state=42)
# Store model in `rf_model`, accuracy in `rf_accuracy`
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train,y_train)
rf_accuracy = accuracy_score(y_test,rf_model.predict(X_test))

print("logreg_accuracy:", logreg_accuracy)
print("dt_accuracy:",dt_accuracy)
print("rf_accuracy:",rf_accuracy)

logreg_accuracy: 0.856
dt_accuracy: 0.778
rf_accuracy: 0.8535


## Task 6: Evaluation

In [36]:
from sklearn.metrics import confusion_matrix, classification_report

# TODO: Determine which model had the best accuracy (from KNN-k5, LogReg, DT, RF)
# Set `best_model_name` to one of: 'KNN', 'LogisticRegression', 'DecisionTree', 'RandomForest'
accuracies={
    'KNN':knn_accuracy_k5,
    'LogisticRegression':logreg_accuracy,
    'DecisionTree':dt_accuracy,
    'RandomForest': rf_accuracy
}
best_model_name =max(accuracies,key=accuracies.get)

preds_lookup={
    'KNN': knn_preds_k5,
    'LogisticRegression':logreg_model.predict(X_test),
    'DecisionTree': dt_model.predict(X_test),
    'RandomForest': rf_model.predict(X_test)
    }
best_preds=preds_lookup[best_model_name]

# TODO: Generate the confusion matrix for the BEST model's predictions on X_test
confusion_mat =confusion_matrix(y_test, best_preds)

# TODO: Generate the classification report as a DICTIONARY for the BEST model
# Use output_dict=True
class_report =classification_report(y_test,best_preds, output_dict=True)

print("best_model_name:",best_model_name)
print("\nconfusion mat:\n",confusion_mat)
print("\nclass_report:\n",class_report)


best_model_name: LogisticRegression

confusion mat:
 [[248  50   0   0]
 [ 36 635  59   0]
 [  0  41 588  46]
 [  0   0  56 241]]

class_report:
 {'A': {'precision': 0.8732394366197183, 'recall': 0.8322147651006712, 'f1-score': 0.852233676975945, 'support': 298.0}, 'B': {'precision': 0.8746556473829201, 'recall': 0.8698630136986302, 'f1-score': 0.8722527472527473, 'support': 730.0}, 'C': {'precision': 0.8364153627311522, 'recall': 0.8711111111111111, 'f1-score': 0.8534107402031931, 'support': 675.0}, 'F': {'precision': 0.8397212543554007, 'recall': 0.8114478114478114, 'f1-score': 0.8253424657534246, 'support': 297.0}, 'accuracy': 0.856, 'macro avg': {'precision': 0.8560079252722979, 'recall': 0.846159175339556, 'f1-score': 0.8508099075463276, 'support': 2000.0}, 'weighted avg': {'precision': 0.8563507785446448, 'recall': 0.856, 'f1-score': 0.8559445515996297, 'support': 2000.0}}


## Task 7: Model Comparison & Cross-Validation

In [37]:
from sklearn.model_selection import cross_val_score

# TODO: Create a DataFrame `comparison_df` with columns: ['Model', 'Accuracy']
# It should contain rows for 'KNN-k5', 'LogisticRegression', 'DecisionTree', and 'RandomForest'
comparison_df = pd.DataFrame({
    'Model':['KNN-k5','LogisticRegression','DecisionTree','RandomForest'],
    'Accuracy':[knn_accuracy_k5,logreg_accuracy,dt_accuracy,rf_accuracy]
})

# TODO: Run 5-fold cross-validation on the Random Forest model using the FULL scaled dataset (X_scaled, y)
# Store the array of 5 scores in `cv_scores`
cv_scores = cross_val_score(
    RandomForestClassifier(n_estimators=100,random_state=42),
    X_scaled,y,cv=5
)

print(comparison_df)
print("\ncv_scores:",cv_scores)
print("mean cv accuracy:",cv_scores.mean())

                Model  Accuracy
0              KNN-k5    0.8340
1  LogisticRegression    0.8560
2        DecisionTree    0.7780
3        RandomForest    0.8535

cv_scores: [0.86  0.85  0.855 0.847 0.849]
mean cv accuracy: 0.8522000000000001
